In [16]:
import os
import time
import base64
from io import BytesIO
from typing import List, Dict, Any, Tuple, Optional, Literal
from llama_index.llms.ollama import Ollama

from PIL import Image, ImageDraw
import nest_asyncio
import dotenv
import regex as re
import matplotlib.pyplot as plt
import pyautogui as pg
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import FunctionAgent, ReActAgent, AgentWorkflow
from llama_index.core.multi_modal_llms.generic_utils import load_image_urls
from llama_index.core.llms import ChatMessage, MessageRole, ImageBlock
from llama_index.core.schema import Document, MediaResource, ImageDocument
from llama_index.llms.openai_like import OpenAILike
from llama_index.multi_modal_llms.openai.utils import (
    generate_openai_multi_modal_chat_message, )
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
)
from IPython.display import Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer

dotenv.load_dotenv()
nest_asyncio.apply()

In [2]:
deepseek = Ollama(model="MFDoom/deepseek-r1-tool-calling",
                  request_timeout=300.0)

In [8]:
# Qwen
qwen_chat = OpenAILike(
    model="qwen-max-latest",
    api_base=os.environ.get("QWEN_API_BASE"),
    api_key=os.environ.get("QWEN_API_KEY"),
    is_chat_model=True,
    is_function_calling_model=True,
)
qwen_vl2 = OpenAILike(
    model="qwen-vl-max",
    api_base=os.environ.get("QWEN_API_BASE"),
    api_key=os.environ.get("QWEN_API_KEY"),
    is_chat_model=True,
    is_function_calling_model=True,
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-VL-Chat",
                                          trust_remote_code=True)

In [14]:
async def screenshot():
    """Useful when you want to take a screenshot of the current screen"""

    img = pg.screenshot()
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img.save("../images/screenshot.png")
    img_str = base64.b64encode(buffered.getvalue())

    return img_str


async def parse_current_screen(ctx: Context):
    """Useful when you want to know the information and status of the current screen"""

    img_str = await screenshot()
    msg = generate_openai_multi_modal_chat_message(
        prompt=
        "Describe the screenshot in detail. Do not miss the task bar at the bottom.",
        role="user",
        image_documents=[ImageDocument(image=img_str)],
    )
    response = await qwen_vl2.achat([msg])

    state = await ctx.get("state")
    state["current_screen_info"] = response.message.content
    state["current_screenshot"] = img_str
    await ctx.set("state", state)

    return response.message.content

In [8]:
# Qwen
image_path = "../images/start.png"
chat_msg_1 = generate_openai_multi_modal_chat_message(
    prompt="Describe the screenshot in detail. Do not miss the task bar at the bottom.",
    role="user",
    image_documents=[ImageDocument(image=await screenshot())],
)
response = await qwen_vl2.achat([chat_msg_1])
display(Markdown(f"<b>{response.message.content}</b>"))

<b>The screenshot shows a Jupyter Notebook interface with Python code being edited. Here are the details:

1. **Notebook Interface**:
   - The notebook is named `workflow.ipynb`.
   - There are two cells visible in the notebook.
   - The first cell contains a function that takes a screenshot, encodes it as a base64 string, and returns the string.
   - The second cell contains an asynchronous function `parse_current_screen` which seems to be part of a larger process involving image processing and possibly machine learning or computer vision tasks.

2. **Code Details**:
   - The first cell's code snippet:
     ```python
     img = pg.screenshot()
     buffered = BytesIO()
     img.save(buffered, format="PNG")
     img_str = base64.b64encode(buffered.getvalue())
     return img_str
     ```
   - The second cell's code snippet:
     ```python
     async def parse_current_screen(ctx: Context):
         """
         Useful when you want to parse the current screen
         """
         img_str = await screenshot()
         msg = generate_openai_multi_modal_chat_message(
             prompt="Describe the screenshot in detail. Do not miss the task bar at the bottom.",
             role="user",
             image_documents=[ImageDocument(image=img_str)],
         )
         response = await qwen_v12.achat([msg])
         state = await ctx.get("state")
         state["current_screen_info"] = response.message.content
         state["current_screenshot"] = img_str
         await ctx.set("state", state)
         return response.message.content
     ```

3. **Task Bar at the Bottom**:
   - The task bar at the bottom of the screen shows several icons and applications:
     - File explorer icon.
     - Microsoft Edge browser icon.
     - Visual Studio Code icon.
     - Task view icon.
     - Start menu icon.
   - The system tray on the right side shows various status icons such as network, battery, and sound.

4. **Other Elements**:
   - On the right side, there is a sidebar with a chat window labeled "GitHub Copilot" where a conversation about Python typing and specific choices for a parameter type is taking place.
   - The top bar includes options like "File," "Edit," "Selection," "View," "Go," "Run," and other notebook controls.
   - The bottom left corner shows some system information like CPU usage, memory usage, and disk activity.

This screenshot appears to be from a development environment where the user is working on a project involving image processing and possibly integrating with AI services like OpenAI.</b>

In [17]:
def special_token_align(text: str):
    text = re.sub(r'object_|\||_start', '', text)
    text = re.sub(r"([a-z]+?)_end", r"/\1", text)
    return text


async def locate_element(ctx: Context, element_name: str):
    """Useful when you want to locate a specific element in the current screen"""

    time.sleep(3)
    state = await ctx.get("state")
    img_str = state.get("current_screenshot")

    if not img_str:
        img_str = await screenshot()

    msg = generate_openai_multi_modal_chat_message(
        prompt=f"框出图中\"{element_name}\"的位置",
        role="user",
        image_documents=[ImageDocument(image=img_str)],
    )
    response = await qwen_vl2.achat([msg])
    boxes = re.findall(r"<\|box_start\|>(.+)<\|box_end\|>",
                       response.message.content)

    if not boxes:
        state["current_screenshot"] = None
        if state.setdefault("retry", 0) < 3:
            state["retry"] += 1
            await ctx.set("state", state)
            return f"Failed to locate the element: {element_name}. Please try again."
        else:
            return f"Failed to locate the element: {element_name}. All retries exhausted. Stop the process."

    state["retry"] = 0
    w, h = 2160, 1440
    x1, y1, x2, y2 = list(map(int, re.findall(r"\d+", boxes[0])))
    x1, y1, x2, y2 = (int(x1 / 1000 * w), int(y1 / 1000 * h),
                      int(x2 / 1000 * w), int(y2 / 1000 * h))
    x, y = (x1 + x2) // 2, (y1 + y2) // 2
    state.setdefault("elements", {})[element_name] = {
        "boxes": ((x1, y1), (x2, y2)),
        "position": (x, y)
    }

    img = Image.open("../images/screenshot.png")
    draw = ImageDraw.ImageDraw(img)
    draw.rectangle(((x1, y1), (x2, y2)), outline="red", width=2)
    img.save("../images/screenshot_box.png")

    return f"Element: {element_name} is located at ({x}, {y}). Done."

In [6]:
async def click(ctx: Context, element_name: str,
                click_mode: Literal["left", "right", "double"]):
    """Useful when you want to click on a specific element in the current screen"""

    state = await ctx.get("state")
    element = state.get("elements", {}).get(element_name)

    if not element:
        return f"Element: {element_name} not found. Please locate it first."

    if click_mode == "double":
        pg.doubleClick(*element["position"])
    else:
        pg.click(*element["position"], button=click_mode)

    return f"Element: {element_name} clicked successfully."

In [18]:
monitor_agent = FunctionAgent(
    name="MonitorAgent",
    description=
    "Useful for monitoring the current screen and locating elements on the screen.",
    system_prompt=(
        "You are the MonitorAgent that can monitor the current screen and locate elements on the screen. "
        "Once you locate an element successfully, you should hand off control to the ClickAgent to click on the element."
        "You should find at least one element before handing off control to the ClickAgent."
        #  "You need to check the screen again once ClickAgent hands off control to you. You need validate the click action is successful or not."
    ),
    llm=qwen_chat,
    tools=[
        # parse_current_screen,
        locate_element,
    ],
    can_handoff_to=["ClickAgent"],
)

click_agent = FunctionAgent(
    name="ClickAgent",
    description="Useful for clicking on elements on the current screen.",
    system_prompt=(
        "You are the ClickAgent that can click on elements on the current screen. "
        #  "Once you click on an element successfully, you should hand off control to the MonitorAgent for the next step."
    ),
    llm=qwen_chat,
    tools=[click],
    # can_handoff_to=["MonitorAgent"],
)

In [19]:
workflow = AgentWorkflow(
    agents=[monitor_agent, click_agent],
    root_agent=monitor_agent.name,
    # initial_state={
    #     "research_notes": {},
    #     "report_content": "Not written yet.",
    #     "review": "Review required.",
    # },
)

In [13]:
workflow = AgentWorkflow.from_tools_or_functions(
    [parse_current_screen, locate_element, click],
    llm=deepseek,
    system_prompt=
    "You are a helpful assistant that can click elements on the screen.",
)

In [20]:
handler = workflow.run(user_msg=("Help me open the application: Bandizip"))
current_agent = None
current_tool_calls = ""
async for event in handler.stream_events():
    if (hasattr(event, "current_agent_name")
            and event.current_agent_name != current_agent):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")

    # if isinstance(event, AgentStream):
    #     if event.delta:
    #         print(event.delta, end="", flush=True)
    # elif isinstance(event, AgentInput):
    #     print("📥 Input:", event.input)
    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")


🤖 Agent: MonitorAgent

🛠️  Planning to use tools: ['locate_element']
🔨 Calling Tool: locate_element
  With arguments: {'element_name': 'Bandizip'}
🔧 Tool Result (locate_element):
  Arguments: {'element_name': 'Bandizip'}
  Output: Element: Bandizip is located at (793, 659). Done.
📤 Output: ![](https://local/generate_screen_capture/1703598642.497995.png?Expires=1703600442&OSSAccessKeyId=Y27Pl4aql42bElMvBwmkEjvF&Signature=V%2BQJLwXsYRg%2F%2BmKZpU%2F%2F%2F%2F%2F%3D)

I have located the Bandizip application on the screen. Now, I will hand off control to the ClickAgent to open it.
🛠️  Planning to use tools: ['handoff']
🔨 Calling Tool: handoff
  With arguments: {'reason': 'To click on the located element', 'to_agent': 'ClickAgent'}
🔧 Tool Result (handoff):
  Arguments: {'reason': 'To click on the located element', 'to_agent': 'ClickAgent'}
  Output: Agent ClickAgent is now handling the request due to the following reason: To click on the located element.
Please continue with the current req

In [ ]:
response = await workflow.run(user_msg="What is the weather in San Francisco?")
print(str(response))

## pyautogui

In [3]:
im1 = pg.screenshot()

In [10]:
buffered = BytesIO()
im1.save(buffered, format="PNG")
img_str = base64.b64encode(buffered.getvalue())

In [11]:
chat_msg_1 = generate_openai_multi_modal_chat_message(
    prompt="描述图片",
    role="user",
    image_documents=[ImageDocument(image=img_str)],
)
response = qwen_vl2.chat([chat_msg_1])
display(Markdown(f"<b>{response.message.content}</b>"))

<b>The image shows a screenshot of a Jupyter Notebook interface running on a computer. The notebook is named `workflow.ipynb` and is open in a development environment, likely a web-based IDE or a local Jupyter server.

### Key Elements in the Image:

1. **Notebook Interface**:
   - The top bar includes options for file operations (File, Edit, Selection, View, Go, Run), as well as buttons for saving, downloading, and other actions.
   - There are also icons for creating new cells, running cells, interrupting kernel, restarting kernel, clearing outputs, and going to specific lines or variables.

2. **Code Cells**:
   - The notebook contains multiple code cells written in Python.
   - One of the cells is highlighted and contains an error message indicating a problem with taking a screenshot using the `pyautogui` library.
   - The error message is an `OSError` stating "screen grab failed."

3. **Error Details**:
   - The traceback points to a line in a file located in the `pyscreeze` package, specifically in `_screenshot_win32.py`.
   - The error occurs when trying to grab the screen using the `ImageGrab.grab()` function from the `PIL` (Python Imaging Library) module.

4. **Environment Information**:
   - The bottom right corner shows the system information: CPU usage (6%), GPU usage (3%), memory usage (47%), and disk usage (0%).
   - The time and date are also displayed: 6:42 PM, 3/4/2023.

5. **Code Snippet**:
   - The code snippet in the cell that caused the error is:
     ```python
     im1 = pg.screenshot()
     ```

6. **Additional Context**:
   - The notebook appears to be part of a workflow involving automation or GUI interaction, as suggested by the use of `pyautogui`.

### Summary:
The image depicts a Jupyter Notebook with a Python script attempting to take a screenshot using `pyautogui`. An error has occurred, and the traceback suggests issues with the `ImageGrab` function from the `PIL` library. The environment is a typical Jupyter setup with some system resource metrics visible.</b>

## Qwen

In [15]:
time.sleep(3)
img = pg.screenshot()
buffered = BytesIO()
img.save(buffered, format="PNG")
img_str = base64.b64encode(buffered.getvalue())
img.save("../images/screenshot.png")

In [ ]:
# img_str = await screenshot()
msg = generate_openai_multi_modal_chat_message(
    prompt=
    "框出图中\"Terminal\"的位置",
    role="user",
    image_documents=[ImageDocument(image=img_str)],
)
response = await qwen_vl2.achat([msg])
display(Markdown(f"<b>{response.message.content}</b>"))

<b>"Terminal"的检测框位于代码编辑器的注释部分。具体位置如下：

1. 在代码编辑器中，找到包含字符串 "The 'Terminal' icon is located in the 'Pinned' section of the start menu." 的注释。
2. 该注释位于代码的第4个代码块中，大约在代码编辑器的中间位置。
3. 具体来说，这段注释在代码块的顶部，描述了如何在开始菜单的"Pinned"部分找到"Terminal"图标的位置。

你可以使用文本搜索功能（通常按Ctrl+F或Cmd+F）来快速定位到包含"Terminal"的注释行。</b>

In [9]:
# Qwen
image_path = "../images/screenshot.png"
chat_msg_1 = generate_openai_multi_modal_chat_message(
    prompt="框出图中\"Bandizip\"的位置",
    role="user",
    image_documents=[ImageDocument(image_path=image_path)],
)
response = await qwen_vl2.achat([chat_msg_1])
display(Markdown(f"<b>{response.message.content}</b>"))

<b><|object_ref_start|>Bandizip<|object_ref_end|><|box_start|>(329,561),(387,604)<|box_end|></b>

In [ ]:
boxes = re.findall(r"<\|box_start\|>(.+)<\|box_end\|>",
                   response.message.content)

In [ ]:
re.findall(r"\d+", boxes[0])

['519', '426', '553', '487']

In [ ]:
response.message.content

'<|object_ref_start|>Zoom<|object_ref_end|><|box_start|>(375,506),(421,589)<|box_end|>'

In [ ]:
text = re.sub(r'object_|\||_start', '', response.message.content)
text = re.sub(r"([a-z]+?)_end", r"/\1", text)
text

'<ref>Zoom</ref><box>(375,506),(421,589)</box>'